# Appendix: Coefficients and Contrasts

The regularization notebook judges every model by how well it predicts new data. A separate question is how well a model *estimates its coefficients*: how close is `β̂` to the true `β`. The two are related, but the connection is somewhat subtle.

This appendix uses the regularization notebook's simulated data, unpenalized fit, and cross-validated ridge fit:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from glm.datasets import make_frame
from glm.families import QuasiPoisson
from glm.glm import GLM
from glm.ridge import RidgePathCV

In [ ]:
n_train = 500
n_test = 20_000

sim = make_frame(
    QuasiPoisson(),
    n_samples=n_train + n_test,
    n_uncorrelated=10,
    n_correlated=50,
    correlation=0.8,
    coef_range=(-0.3, 0.3),
    latent_noise_sd=1.0,
    rng=np.random.default_rng(11),
)
train = sim.data.iloc[:n_train]
test = sim.data.iloc[n_train:]
formula = sim.formula

unpenalized = GLM(QuasiPoisson()).fit(train, formula=formula)
cv = RidgePathCV(QuasiPoisson(), n_folds=10).fit(
    train, formula=formula, rng=np.random.default_rng(6)
)

## Unpenalized estimates

Since the data are simulated, we know the true coefficients, and can compare the unpenalized estimates with them. Let's scatterplot the estimated vs. true coefficients:

In [ ]:
def plot_estimates(ax, estimates, truth, label, lim=0.75):
    """Estimated coefficients against the true ones, without the intercept,
    on axes from -lim to lim."""
    ax.axline((0, 0), slope=1, color="grey", linestyle="--", linewidth=1)
    ax.scatter(truth[1:], estimates[1:], color="C1", label=label)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("true coefficient")
    ax.set_ylabel("estimated coefficient")
    ax.legend()
    return ax


fig, ax = plt.subplots(figsize=(5, 5))
plot_estimates(ax, unpenalized.coef_, sim.coef.to_numpy(), "unpenalized")
ax.set_title("Unpenalized Estimates");

The estimates scatter widely around the truth due to variance in estimation. This variance of the estimated coefficient around it's expected value is captured by our standard errors and their associated intervals:

In [ ]:
def plot_coefficients(ax, estimates, standard_errors, truth=None, n_standard_errors=2):
    """Each estimate against its index, from 1, with bars of n_standard_errors
    standard errors either side, and the true values if given."""
    index = np.arange(1, len(estimates) + 1)
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.errorbar(
        index,
        estimates,
        yerr=n_standard_errors * standard_errors,
        fmt="o",
        color="C1",
        markersize=5,
        elinewidth=1,
        label=f"estimate ± {n_standard_errors} standard errors",
    )
    if truth is not None:
        ax.scatter(index, truth, color="black", marker="o", label="true value")
    ax.set_xlabel("coefficient index")
    ax.set_ylabel("coefficient")
    ax.legend()
    return ax


fig, ax = plt.subplots(figsize=(14, 4))
plot_coefficients(
    ax,
    unpenalized.coef_[1:],
    unpenalized.coef_standard_error_[1:],
    sim.coef.to_numpy()[1:],
)
ax.set_title("Unpenalized Estimates");

The standard errors are large, on average larger than the size of an estimated coefficient, so many estimates are within two standard errors of zero. This poor estimation indicates our model specification is quite sensitive to the particular training data we happened to draw.

In [ ]:
print(f"Typical size of a true coefficient: {np.abs(sim.coef[1:]).mean():.3f}")
print(f"Typical standard error:             {unpenalized.coef_standard_error_[1:].mean():.3f}")

## Correlated predictors and contrasts

A close look at the interval plot reveals that the standard errors form two groups. The first ten intervals are "narrow" and the remaining are "wide".

In [ ]:
print(f"Typical standard error:             {unpenalized.coef_standard_error_[1:11].mean():.3f}")
print(f"Typical standard error:             {unpenalized.coef_standard_error_[11:].mean():.3f}")

This is due to the correlation structure in our data. Reacall our generation parameters:

```
sim = make_frame(
    ...
    n_uncorrelated=10,
    n_correlated=50,
    correlation=0.8,
    ...
)
```

We ensured that the first ten predictors are statistically independent, but the remaining are mutually strongly correlated. Correlation structure tends to increase estimation variance. Thinking about the case of only two predictors is illuminating.

When `x11` and `x12` mostly move together, the data says a lot about their *combined* effect, but little about how to split it between them. In the extreme case of two **identical** predictors, there is a one-parameter family of coefficients that all lead to the same predictions:

```
γ x11 + (β - γ) x12 = β x11, for all γ
```

A linear combination of the coefficients, `c₁β₁ + c₂β₂ + ⋯ + cp βp`, is called a **contrast** when its multipliers sum to zero: `c₁ + c₂ + ⋯ + c_p = 0`. The simplest kind is a difference such as `β₁₁ - β₁₂`. In the setup above:

```
β₁₁ - β₁₂ = γ - β + γ = 2γ - β
```

So the contrast can be anything, it is not determined by the data at all. Correlated predictors have a fuzzy verion of this behaviour: contrasts between correlated predictors are exactly what the data struggle to estimate precisely. 

Back to our simulated example data, among the 50 predictors correlated at 0.8, there are 49 linearly independent contrasts which are all poorly determined, for example the differences `β₁₁ - β₁₂`, `β₁₂ - β₁₃`, …, `β₅₉ - β₆₀` (any basis for this span works). The one remaining independent direction, their average effect, is what the data determines very well.

We show this visually by computing the contrasts and their standard errors for the basis above. If `V` is the coefficient covariance matrix, the standard error of any linear combination of the coefficients `cᵗβ̂`, in particular any constrast, is `√(cᵗVc)`. We scale to unit length, `Σc² = 1`, so that the standard errors are comparable with each other, and with those of the individual coefficient estiamtes. (Without the scaling, the mean of 50 coefficients would look precise simply from averaging, correlated or not.)

In [ ]:
# Unit-length directions in coefficient space, one per row, so that their
# standard errors are comparable.
n_coef = len(unpenalized.coef_)
directions = []
# x1 to x10, each alone.
for j in range(1, 11):
    direction = np.zeros(n_coef)
    direction[j] = 1
    directions.append(direction)
# The average of x11 to x60.
direction = np.zeros(n_coef)
direction[11:] = 1 / np.sqrt(50)
directions.append(direction)
# The differences between neighbours, x11 - x12 to x59 - x60.
for j in range(11, 60):
    direction = np.zeros(n_coef)
    direction[j] = 1 / np.sqrt(2)
    direction[j + 1] = -1 / np.sqrt(2)
    directions.append(direction)
directions = np.array(directions)

estimates = directions @ unpenalized.coef_
truth = directions @ sim.coef.to_numpy()
covariance = directions @ unpenalized.coef_covariance_matrix_ @ directions.T
standard_errors = np.sqrt(np.diag(covariance))

print(f"x1 to x10, each alone:  {standard_errors[:10].min():.3f} to {standard_errors[:10].max():.3f}")
print(f"average of x11 to x60:  {standard_errors[10]:.3f}")
print(f"differences:            {standard_errors[11:].min():.3f} to {standard_errors[11:].max():.3f}")

fig, ax = plt.subplots(figsize=(14, 4))
plot_coefficients(ax, estimates, standard_errors, truth)
for boundary in (10.5, 11.5):
    ax.axvline(boundary, color="gold", linewidth=0.5, linestyle='--')
ax.axvspan(10.5, 11.5, color="yellow", linewidth=0.5, linestyle='--', alpha=0.33)

for x, label in [(5.5, "x1 to x10"), (11, "average"), (36, "differences of neighbours, x11 - x12 to x59 - x60")]:
    ax.text(x, 0.97, label, ha="center", va="top", transform=ax.get_xaxis_transform())
ax.set_xlabel("direction")
ax.set_ylabel("unit-length combination")
ax.set_title("Unpenalized Estimates Along Unit-Length Directions");

This plot is very similar to the original coefficient plot, with one important difference, highlighted in yellow. The "average coefficient" contrast is determined very precisely. The other 59 directions, those spanned by the differences of neighbours, are detemined about as weakly as the original coefficients.

How does this make the individual coefficients poorly determined? Each coefficient in the correlated block is the block's average plus its own deviation from that average, `β₁₁ = β̄ + (β₁₁ - β̄)`, and that deviation is a contrast. The data pin the average down, but say little about the contrast `β₁₁ - β̄`. So each individual coefficient inherits the large standard error of the contrasts.

That's what the coefficient plot showed: the standard errors on the right, for the correlated `x11` to `x60`, are larger than those on the left, for the independent `x1` to `x10`.

## Poor coefficients don't imply poor predictions

It's tempting to read those large standard errors as the reason the unpenalized model predicts poorly, but that isn't right. 

An error in the coefficient estimation only hurts predictions to the extent that it changes the linear predictor `β₁x₁ + β₂x₂ + ...` of a new datapoint.

An error along a contrast of correlated predictors hardly moves the linear predictor (recall our extreme example of two equal predictors), so the coefficients are worst estimated in exactly the directions where errors matter least for prediction.


For example, suppose the estimate of `β₁₁` is too high by `δ`, and that of `β₁₂` too low by `δ`: an error purely in the contrast `β₁₁ - β₁₂`. On new data, the linear predictor moves by:

```
δ x11 - δ x12 = δ (x11 - x12)
```

Since `x11` and `x12` are correlated at `0.8`, their difference varies little from one observation to the next. Its standard deviation is

```
sd(x11 - x12) = √(2 - 2 × 0.8) ≈ 0.63
``` 
compared to 

```
sd(x1 - x2) = √2 ≈ 1.41
``` 

for two independent predictors. So the same error moves the predictions only `√(1 - 0.8) ≈ 0.45` times as much.

That relative prediction error exactly offsets the contrast's larger standard error, which was ≈ 2.24 times that of an independent coefficient, since 2.24 × 0.45 = 1. The same balance holds for the average, which is estimated about six times more precisely, but moves the predictions about six times as much as an independent predictor. In general: every direction contributes equally to the error in the predictions, the difficulty of estimating that direction balances with how sensitive the predictions are to errors in that direction.

For a linear model this can be made exact: the expected excess prediction error of least squares is about `σ²p / n`, however correlated the predictors are. The unpenalized model predicts poorly because it estimates 60 coefficients from 500 noisy, overdispersed counts, not because its predictors are correlated.

## Ridge and the contrasts

Now we can say precisely what the penalty does to the coefficients. Ridge adds `α` to the information along every direction alike: the penalized fit uses `H + αI` in place of the information matrix `H`. Along a direction where the data carry information `d = cᵗHc`, the same information that set its standard error, the unpenalized estimate is shrunk by the factor `d / (d + α)`. (Exactly so for the eigenvectors of `H`, which our directions roughly are, and approximately for a GLM, whose weights change as the fit does.) Directions the data determine well, with `d` much larger than `α`, are left nearly alone, and directions they determine poorly are shrunk hard.

Here are the information along our directions, the shrinkage it predicts, and the shrinkage ridge actually applies at the cross-validated `alpha_min_`:

In [ ]:
ridge = cv.model_min_
ridge_estimates = directions @ ridge.coef_

# The information along each unit-length direction, cᵗHc.
information = np.diag(directions @ unpenalized.information_matrix_ @ directions.T)
predicted_shrinkage = information / (information + cv.alpha_min_)


def root_mean_square(x):
    return np.sqrt(np.mean(x**2))


groups = {
    "x1 to x10, each alone": slice(0, 10),
    "average of x11 to x60": slice(10, 11),
    "differences": slice(11, None),
}
pd.DataFrame(
    {
        name: {
            "information": information[group].mean(),
            "d / (d + alpha)": predicted_shrinkage[group].mean(),
            # The least squares slope, through the origin, of the ridge
            # estimates on the unpenalized ones.
            "ridge / unpenalized": (
                np.sum(ridge_estimates[group] * estimates[group])
                / np.sum(estimates[group] ** 2)
            ),
            "unpenalized error": root_mean_square(estimates[group] - truth[group]),
            "ridge error": root_mean_square(ridge_estimates[group] - truth[group]),
        }
        for name, group in groups.items()
    }
).T.round(3)

The average of the correlated block carries about fifty times the information of an independent predictor, and is barely shrunk. Each difference carries about a fifth as much, and is shrunk to about a third. Ridge does almost all of its work on the contrasts. Here are its estimates along our directions, beside the unpenalized ones:

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_coefficients(ax, estimates, standard_errors, truth)
ax.scatter(
    np.arange(1, len(ridge_estimates) + 1),
    ridge_estimates,
    color="C0",
    zorder=3,
    label="ridge estimate",
)
ax.axvspan(10.5, 11.5, color="yellow", alpha=0.33)
ax.set_xlabel("direction")
ax.set_ylabel("unit-length combination")
ax.legend()
ax.set_title("Unpenalized and Ridge Estimates Along Unit-Length Directions");

correlated_truth = sim.coef.to_numpy()[11:]
correlated_unpenalized = unpenalized.coef_[11:]
correlated_ridge = ridge.coef_[11:]
print("Correlated coefficients, x11 to x60:")
print(f"  mean:          truth {correlated_truth.mean():.4f}, unpenalized {correlated_unpenalized.mean():.4f}, ridge {correlated_ridge.mean():.4f}")
print(f"  spread (sd):   truth {correlated_truth.std():.3f},  unpenalized {correlated_unpenalized.std():.3f},  ridge {correlated_ridge.std():.3f}")

Since ridge leaves the block's average alone, and shrinks each coefficient's deviation from it, it pulls the correlated coefficients toward *each other*, rather than toward zero: their mean is unchanged, and their spread is cut by more than two thirds.

Why is that a good trade for prediction? Shrinking a direction's estimate by a factor `s` cuts its variance by `s²`, and we saw that every direction's variance contributes the same amount to the prediction error, so that saving is the same wherever we shrink. bias.

The errors in the table show the trade on the coefficients themselves. Ridge cuts the error of the differences from about 0.20 to 0.12, and leaves the average's about where it was. It makes the independent coefficients slightly *worse*, from 0.091 to 0.100: there its bias isn't cheap, and costs more than the variance it saves.

## Ridge estimates

The penalized model's coefficients are closer to the truth, shrunk toward zero, and no longer swamped by noise. Here are the estimates on the same scales, and the root mean squared error of each model's coefficients:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
plot_estimates(axs[0], unpenalized.coef_, sim.coef.to_numpy(), "unpenalized")
plot_estimates(axs[1], cv.model_min_.coef_, sim.coef.to_numpy(), "ridge, alpha_min_")
axs[0].set_title("Unpenalized Estimates")
axs[1].set_title("Ridge Estimates");

def coefficient_rmse(model):
    return np.sqrt(np.mean((model.coef_[1:] - sim.coef.to_numpy()[1:]) ** 2))

print(f"Unpenalized coefficient RMSE: {coefficient_rmse(unpenalized):.3f}")
print(f"Ridge coefficient RMSE:       {coefficient_rmse(cv.model_min_):.3f}")